# BSRGAN آزمایشی برای بهبود خواندن متن کارت

این نوت‌بوک از شاخهٔ مستقل `feat/bsrgan-ocr-experiment` استفاده می‌کند. با فعال‌کردن BSRGAN، تصویر x2 هم برای پیدا کردن جعبه‌های متن و هم برای خواندن cropها با OCR استفاده می‌شود.

برای هر فیلد خروجی تصویر اصلی و خروجی BSRGAN ثبت می‌شود. مقدار BSRGAN فقط وقتی انتخاب می‌شود که قالبش با نوع فیلد سازگار باشد؛ در غیر این صورت خروجی اصلی می‌ماند. هر دو مقدار در گزارش مقایسه نگه داشته می‌شوند.

جعبه‌های تازه در `text_boxes.png` نارنجی هستند و تصویر x2 در `bsrgan_card_x2.png` ذخیره می‌شود. BSRGAN آزمایشی است و ممکن است جزئیات مصنوعی بسازد؛ نتیجه را با تصویر اصلی بسنجید.


## دریافت نسخهٔ یکپارچه
سلول بعد سورس یک commit مشخص و اصلاح‌شده را مستقیم دریافت می‌کند و به API گیت نیاز ندارد. شناسهٔ نسخه در خروجی ذخیره می‌شود.


In [ ]:
from google.colab import files
from pathlib import Path
import json
import subprocess
import sys
import shutil
import tempfile
import zipfile
import urllib.request

REPOSITORY = 'neginnoori/Persian_OCR_IDCard'
BRANCH = 'feat/bsrgan-ocr-experiment'
# Pinned source bundle: direct commit archive; no GitHub API request.
SOURCE_COMMIT = 'f38948e12e4f92214b5634e428289ec345e5a912'
SOURCE = Path(tempfile.mkdtemp(prefix='persian-id-source-', dir='/content'))
archive_path = SOURCE / 'source.zip'
url = f'https://codeload.github.com/{REPOSITORY}/zip/{SOURCE_COMMIT}'
with urllib.request.urlopen(url, timeout=120) as response:
    archive_path.write_bytes(response.read())
with zipfile.ZipFile(archive_path) as archive:
    if sum(m.file_size for m in archive.infolist()) > 100 * 1024 * 1024:
        raise ValueError('Source archive is unexpectedly large')
    for member in archive.infolist():
        target = (SOURCE / member.filename).resolve()
        if not target.is_relative_to(SOURCE.resolve()):
            raise ValueError('Invalid archive path')
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Source archive must not contain symlinks')
    archive.extractall(SOURCE)
projects = [p.parent for p in SOURCE.rglob('pyproject.toml')
            if (p.parent / 'scripts/setup_colab.py').is_file()
            and (p.parent / 'src/persian_id_ocr').is_dir()]
if len(projects) != 1:
    raise RuntimeError('Combined OCR project not found; upload the complete project ZIP')
PROJECT = projects[0]
print('Project:', PROJECT)
print('Source commit:', SOURCE_COMMIT)


## نصب و کنترل نسخه
این سلول کد محلی را حتی با شمارهٔ نسخهٔ یکسان دوباره نصب می‌کند و هش تمام ماژول‌های نصب‌شده را با سورس مقایسه می‌کند. پس از دریافت نسخهٔ تازه، این سلول را دوباره اجرا کنید. مدل‌ها در کش باقی می‌مانند.


In [ ]:
CANDIDATE_SELECTION = 'v3'  # 'v2': نسخهٔ مبنا؛ legacy: مسیر اولیه
PREPROCESSING = 'rectify_best_effort'  # rectify: strict; orientation_only: rotate full photo only
TEXT_DETECTOR = 'paddle'  # comparison option: craft
PADDLE_DEVICE = 'cpu'  # detection only; DINO/SAM/EasyOCR/Hezar remain on T4 GPU
BSRGAN_TEXT_DETECTION = False  # opt-in BSRGAN x2 detection + OCR comparison; downloads weights
BSRGAN_SOURCE_COMMIT = '5ce1a9c6ae292f30ccfce4b597ecb73c70401733'
subprocess.run([sys.executable, str(PROJECT / 'scripts/setup_colab.py'), '--paddle-device', PADDLE_DEVICE], check=True)
PYTHON = '/content/persian-id-venv/bin/python'
subprocess.run([PYTHON, '-u', str(PROJECT / 'scripts/verify_install.py'), str(PROJECT)], check=True)


## آپلود عکس‌های کارت
جهت به‌صورت خودکار روی کارت صاف‌شده بررسی می‌شود. ORIENTATION_OVERRIDE را روی None نگه دارید.
در مسیر rectify، override ترتیب گوشه‌های کارت صاف‌شده را مشخص می‌کند؛ زاویهٔ چرخش نسبت به عکس اصلی نیست. در orientation_only زاویهٔ پادساعت‌گرد نسبت به ورودی اصلاح‌شده با EXIF است.


In [ ]:
photos = files.upload()
INPUT_DIR = Path(tempfile.mkdtemp(prefix='persian-id-inputs-', dir='/content'))
INPUTS = []
for index, (name, content) in enumerate(photos.items()):
    destination = INPUT_DIR / f'{index:03d}_{Path(name).name}'
    destination.write_bytes(content)
    INPUTS.append(str(destination))
if not INPUTS:
    raise ValueError('Upload at least one card photo')
ORIENTATION_OVERRIDE = None  # None (خودکار) یا 0، 90، 180، 270


## پردازش و ثبت تصمیم‌ها
در v3 ابتدا خروجی اولیه تولید می‌شود؛ سپس پیشنهادهای عنوان‌محور و بازخوانی موضعی بررسی می‌شوند.
baseline_fields و تصمیم‌های جایگزینی در metadata.adaptive_fields ذخیره می‌شوند.
اجراهای v2 و v3 در پوشه‌های جدا هستند. اختلاف نام معتبر فقط پیشنهاد بازبینی است.


In [ ]:
subprocess.run([PYTHON, '-u', str(PROJECT / 'scripts/verify_install.py'), str(PROJECT)], check=True)
OUTPUT_ROOT = Path(tempfile.mkdtemp(prefix='persian-id-results-', dir='/content'))
command = [PYTHON, str(PROJECT / 'scripts/run_batch.py'), *INPUTS,
           '--output-root', str(OUTPUT_ROOT), '--cache-dir', '/content/persian-id-model-cache']
if ORIENTATION_OVERRIDE is not None:
    command += ['--orientation-override', str(ORIENTATION_OVERRIDE)]
command += ['--text-detector', TEXT_DETECTOR, '--paddle-device', PADDLE_DEVICE,
            '--preprocessing', PREPROCESSING, '--candidate-selection', CANDIDATE_SELECTION]
BSRGAN_MODEL = None
BSRGAN_SOURCE_DIR = None
if BSRGAN_TEXT_DETECTION and TEXT_DETECTOR != 'paddle':
    raise ValueError('The BSRGAN experiment currently uses the Paddle text detector')
if BSRGAN_TEXT_DETECTION:
    BSRGAN_CACHE = Path('/content/persian-id-model-cache/bsrgan')
    BSRGAN_CACHE.mkdir(parents=True, exist_ok=True)
    BSRGAN_REPO = BSRGAN_CACHE / f'BSRGAN-{BSRGAN_SOURCE_COMMIT}'
    BSRGAN_ARCHIVE = BSRGAN_CACHE / 'bsrgan-source.zip'
    if not (BSRGAN_REPO / 'models/network_rrdbnet.py').is_file():
        if BSRGAN_REPO.exists():
            shutil.rmtree(BSRGAN_REPO)
        unpack_root = BSRGAN_CACHE / 'source-unpack'
        if unpack_root.exists():
            shutil.rmtree(unpack_root)
        unpack_root.mkdir()
        urllib.request.urlretrieve(
            f'https://codeload.github.com/cszn/BSRGAN/zip/{BSRGAN_SOURCE_COMMIT}', BSRGAN_ARCHIVE)
        with zipfile.ZipFile(BSRGAN_ARCHIVE) as archive:
            for member in archive.infolist():
                target = (unpack_root / member.filename).resolve()
                if not target.is_relative_to(unpack_root.resolve()):
                    raise ValueError('Invalid BSRGAN source archive')
                if (member.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError('BSRGAN source archive must not contain symlinks')
            archive.extractall(unpack_root)
        extracted = next(unpack_root.glob('BSRGAN-*/models/network_rrdbnet.py')).parent.parent
        extracted.rename(BSRGAN_REPO)
    BSRGAN_SOURCE_DIR = BSRGAN_REPO
    BSRGAN_MODEL = BSRGAN_CACHE / 'BSRGANx2.pth'
    if not BSRGAN_MODEL.is_file() or BSRGAN_MODEL.stat().st_size != 66894171:
        weight_download = BSRGAN_CACHE / 'BSRGANx2.pth.download'
        if weight_download.exists():
            weight_download.unlink()
        urllib.request.urlretrieve(
            'https://github.com/cszn/KAIR/releases/download/v1.0/BSRGANx2.pth', weight_download)
        if weight_download.stat().st_size != 66894171:
            weight_download.unlink()
            raise RuntimeError('Downloaded BSRGANx2 weight has an unexpected size')
        weight_download.replace(BSRGAN_MODEL)
    command += ['--bsrgan-model', str(BSRGAN_MODEL),
                '--bsrgan-source-dir', str(BSRGAN_SOURCE_DIR)]
completed = subprocess.run(command)
runs = list(OUTPUT_ROOT.glob('run_*'))
if len(runs) != 1 or not (runs[0] / 'summary.json').is_file():
    raise RuntimeError('Run did not complete; inspect the error above')
RESULTS = runs[0]
SUMMARY = json.loads((RESULTS / 'summary.json').read_text())
(RESULTS / 'source_version.json').write_text(json.dumps({'repository': REPOSITORY, 'commit': SOURCE_COMMIT, 'candidate_selection': CANDIDATE_SELECTION, 'bsrgan_text_detection': BSRGAN_TEXT_DETECTION, 'bsrgan_ocr_enhancement': BSRGAN_TEXT_DETECTION}))
print('Source commit:', SOURCE_COMMIT)
print(json.dumps(SUMMARY, ensure_ascii=False, indent=2))
if completed.returncode == 1:
    print('Runtime error in one or more images. Read the JSON reports below; OCR did not succeed for those images.')
elif completed.returncode == 2:
    print('Some images need review, contain no text, or have no detected card.')
elif completed.returncode != 0:
    raise RuntimeError(f'Unexpected process exit: {completed.returncode}')


In [ ]:
from IPython.display import display, Image as DisplayImage
for item in SUMMARY:
    folder = RESULTS / item['folder']
    report = json.loads((folder / 'result.json').read_text())
    print('\n', item['input'], '→', report['status'])
    for filename in ('candidate_boxes.png', 'card.png', 'card_needs_review.png', 'bsrgan_card_x2.png', 'text_boxes.png', 'field_boxes.png'):
        if (folder / filename).is_file():
            print(filename)
            display(DisplayImage(filename=str(folder / filename)))
    extraction_meta = report.get('metadata', {}).get('extraction', {}).get('metadata', {})
    print('Preprocessing:', report.get('metadata', {}).get('preprocessing_mode'))
    print('Effective preprocessing:', report.get('metadata', {}).get('effective_preprocessing'))
    if 'rotation_ccw_degrees' in extraction_meta:
        print('Rotation CCW:', extraction_meta['rotation_ccw_degrees'])
    else:
        print('Rectified card corner shift:', extraction_meta.get('corner_shift_degrees'))
    print('Warnings:', report.get('metadata', {}).get('warnings', []))
    selection = extraction_meta.get('candidate_selection', {})
    print('Card selection:', selection.get('reason'), '| replaced:', selection.get('replaced'))
    print('Field assignment:', report.get('metadata', {}).get('field_assignment'))
    print('Text detection enhancement:', report.get('metadata', {}).get('text_detection_enhancement'))
    recovery = extraction_meta.get('stamp_recovery', {})
    if recovery.get('attempted'):
        print('Stamp recovery:', recovery.get('reason'), '| replaced:', recovery.get('replaced'))
    print('متن خام / Raw text:')
    print(report.get('text', ''))
    if report.get('error'):
        print(report['error'])
    if report['status'] in ('needs_review', 'not_found'):
        print(json.dumps(report.get('metadata', {}).get('extraction', {}), ensure_ascii=False, indent=2))

    import pandas as pd
    display(pd.DataFrame(report.get('fields', [])))
    bsrgan_comparison = report.get('metadata', {}).get('bsrgan_ocr_comparison', {})
    if bsrgan_comparison:
        print('Original OCR vs BSRGAN OCR field comparison:')
        display(pd.DataFrame(bsrgan_comparison.get('decisions', [])))
    repair = report.get('metadata', {}).get('adaptive_fields', {})
    if repair:
        print('Adaptive decisions:')
        display(pd.DataFrame(repair.get('decisions', [])))
        print('Row recovery:', json.dumps(repair.get('row_recovery', {}), ensure_ascii=False, indent=2))
        print('Local reread suggestions:', json.dumps(repair.get('rereads', []), ensure_ascii=False, indent=2))


In [ ]:
# اختیاری: همان عکس‌ها با مسیر قبلی، در پوشهٔ مجزا؛ نتایج v2 حذف نمی‌شوند.
RUN_BASELINE_COMPARISON = True
if RUN_BASELINE_COMPARISON:
    baseline_root = Path(tempfile.mkdtemp(prefix='persian-id-baseline-', dir='/content'))
    baseline_command = command.copy()
    baseline_command[baseline_command.index('--output-root') + 1] = str(baseline_root)
    baseline_command[baseline_command.index('--candidate-selection') + 1] = 'v2'
    if '--bsrgan-model' in baseline_command:
        baseline_command = baseline_command[:baseline_command.index('--bsrgan-model')]
    baseline_run = subprocess.run(baseline_command)
    baseline_runs = list(baseline_root.glob('run_*'))
    if len(baseline_runs) != 1 or not (baseline_runs[0]/'summary.json').is_file():
        raise RuntimeError('Baseline failed; see execution log')
    BASELINE_RESULTS = baseline_runs[0]
    baseline_summary = json.loads((BASELINE_RESULTS/'summary.json').read_text())
    baseline_by_input = {x['input']: x for x in baseline_summary}
    comparison = []
    for item in SUMMARY:
        current = json.loads((RESULTS/item['folder']/'result.json').read_text())
        other = baseline_by_input[item['input']]
        previous = json.loads((BASELINE_RESULTS/other['folder']/'result.json').read_text())
        before = {f['field']:f for f in previous.get('fields', [])}
        after = {f['field']:f for f in current.get('fields', [])}
        for field in ('national_id','first_name','last_name','date_of_birth','father_name','expiry_date'):
            comparison.append({'input':Path(item['input']).name, 'field':field,
                'v2_value':before.get(field,{}).get('value'), 'selected_value':after.get(field,{}).get('value'),
                'v2_field_status':before.get(field,{}).get('status', 'missing'),
                'selected_field_status':after.get(field,{}).get('status', 'missing')})
    import pandas as pd
    print('Selected mode:', CANDIDATE_SELECTION)
    display(pd.DataFrame(comparison))
    (RESULTS/'comparison.json').write_text(json.dumps(comparison,ensure_ascii=False,indent=2))
    print('Baseline results:', BASELINE_RESULTS)
    print('Selected-mode results:', RESULTS)
    print('تفاوت خروجی به معنی درست‌تر بودن نیست؛ با تصویر اصلی مقایسه کنید.')


## دانلود نتایج
ZIP شامل متن UTF-8، متن نرمال‌شده، تصویر کارت، تصویر BSRGAN، کادرهای متن و گزارش JSON است. `ocr_candidate` به معنی صحت قطعی نیست. متن خام حفظ می‌شود و فیلدهای هویتی حدس زده نمی‌شوند.

In [ ]:
import shutil
result_zip = shutil.make_archive(str(RESULTS), 'zip', RESULTS)
files.download(result_zip)


## برگشت
CANDIDATE_SELECTION را v2 کنید و پردازش را دوباره اجرا کنید؛ نصب مجدد لازم نیست. پیش‌فرض کتابخانه legacy باقی مانده است. راهنما: ADAPTIVE_V3.fa.md
